# 09 — Hiệu chỉnh Thiên lệch Độ phổ biến bằng Tô-pô Lân cận Differentiable (Topology-Calibrated SASRec)

## Đóng góp nghiên cứu cốt lõi:
1. **Phân biệt Item A (Nhiễu ngẫu nhiên) vs Item B (Ngọc thô bị vùi dập):** Cả 2 đều ít tương tác, nhưng Item B có cấu trúc lân cận bền vững với Session.
2. **Differentiable Topological Persistence Loss:** Can thiệp trực tiếp vào gradient học biểu diễn của SASRec.
3. **Anisotropy Regularizer:** Đập tan hố đen hấp dẫn của Item Hot, đưa không gian về dạng cầu đẳng hướng.

In [1]:
# ========================================================================
# 0. Cài đặt môi trường làm việc trên Kaggle GPU
# ========================================================================
import os
import shutil
import sys
from pathlib import Path

if Path('/kaggle/input').exists():
    print('[Kaggle Environment Detected]')
    print('Contents of /kaggle/input:', list(Path('/kaggle/input').iterdir()))
    source_dir = None
    for p in Path('/kaggle/input').rglob('BTL-K03-tong-quan-de-tai.md'):
        source_dir = p.parent
        break
    if source_dir is not None:
        print(f'Nguồn dự án tìm thấy tại: {source_dir}')
        working_dir = Path('/kaggle/working')
        for item in source_dir.iterdir():
            target = working_dir / item.name
            if item.is_dir():
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                shutil.copy2(item, target)
        os.chdir(str(working_dir))
        print(f'Thư mục làm việc hiện tại: {Path.cwd()}')
        print('Danh sách tệp tại working dir:', [p.name for p in Path.cwd().iterdir()])
    else:
        raise FileNotFoundError('Không tìm thấy BTL-K03-tong-quan-de-tai.md trong /kaggle/input!')
else:
    print('[Local Environment] Chạy trên thư mục cục bộ.')


[Kaggle Environment Detected]
Contents of /kaggle/input: [PosixPath('/kaggle/input/datasets')]
Nguồn dự án tìm thấy tại: /kaggle/input/datasets/ngytfd/btl-k03-sasrec-data
Thư mục làm việc hiện tại: /kaggle/working
Danh sách tệp tại working dir: ['__notebook__.ipynb', 'configs', 'data', 'BTL-K03-tong-quan-de-tai.md', 'src']


In [ ]:
# ========================================================================
# 1. ĐỊNH NGHĨA MODULE TOPOLOGICAL PERSISTENCE & ANISOTROPY REGULARIZER
# ========================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class TopologicalPersistenceLoss(nn.Module):
    """
    Differentiable Topological Persistence Regularization Loss.
    - Ép khoảng cách trắc địa tô-pô giữa Session và Item B (ngọc thô) nhỏ hơn ngưỡng tau.
    - Phạt hiện tượng mất đẳng hướng (Anisotropy) của các Item Hot chiếm dụng hố đen trung tâm.
    """
    def __init__(self, tau: float = 0.45, lambda_anisotropy: float = 0.05, tail_threshold: int = 20):
        super().__init__()
        self.tau = tau
        self.lambda_anisotropy = lambda_anisotropy
        self.tail_threshold = tail_threshold

    def forward(self, session_vectors: torch.Tensor, target_embeddings: torch.Tensor, 
                item_frequencies: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
        batch_size = session_vectors.size(0)
        if batch_size <= 1:
            return torch.tensor(0.0, device=session_vectors.device, requires_grad=True)

        s_norm = F.normalize(session_vectors, p=2, dim=-1)
        t_norm = F.normalize(target_embeddings, p=2, dim=-1)

        target_freqs = item_frequencies[target_ids]
        tail_mask = target_freqs < self.tail_threshold

        cos_sim = torch.sum(s_norm * t_norm, dim=-1)
        topo_distance = 1.0 - cos_sim

        if tail_mask.sum() > 0:
            tail_topo_loss = F.relu(topo_distance[tail_mask] - self.tau).mean()
        else:
            tail_topo_loss = torch.tensor(0.0, device=session_vectors.device)

        corr_matrix = torch.matmul(s_norm, s_norm.t())
        identity = torch.eye(batch_size, device=session_vectors.device)
        anisotropy_loss = torch.norm(corr_matrix - identity, p='fro') / (batch_size * batch_size)

        return tail_topo_loss + self.lambda_anisotropy * anisotropy_loss

def compute_anisotropy_spectrum(item_embeddings: np.ndarray) -> dict:
    mean_vec = np.mean(item_embeddings, axis=0, keepdims=True)
    centered = item_embeddings - mean_vec
    _, s, _ = np.linalg.svd(centered, full_matrices=False)
    variance_explained = (s ** 2) / np.sum(s ** 2)
    return {
        'top1_singular_ratio': float(variance_explained[0]),
        'top5_singular_ratio': float(np.sum(variance_explained[:5])),
        'singular_values': s[:10].tolist()
    }
print('Đã nạp thành công Topological Module!')


In [ ]:
# ========================================================================
# 2. THIẾT LẬP DỰ ÁN, LOAD DỮ LIỆU VÀ CẤU HÌNH
# ========================================================================
import sys
import time
import copy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'BTL-K03-tong-quan-de-tai.md').exists():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc dự án BTL-K03.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from btl_k03.paths import FIGURES_DIR, ITEM_LEVEL_DIR, PROCESSED_DIGI_DIR, RUNS_DIR, TABLES_DIR, ensure_output_dirs, get_run_dir
from btl_k03.utils import load_experiment_config, load_item_mapping, set_global_seed, save_json, save_yaml, create_run_logger
from btl_k03.models import build_model_from_config, SessionPrefixDataset, predict_topk_for_dataset, build_top20_dataframe
from btl_k03.embedding_analysis import build_item_level_analysis_table
from btl_k03.popularity_metrics import (
    compute_ranking_metrics_from_predictions, 
    compute_stratified_ranking_metrics, 
    evaluate_top20_popularity_bias,
    compute_cold_start_test_diagnostic
)

ensure_output_dirs()
CONFIG = load_experiment_config('sasrec')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device sử dụng: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')


In [ ]:
# ========================================================================
# 3. ENGINE HUẤN LUYỆN SASREC + TOPOLOGY LOSS
# ========================================================================
def run_training_sasrec_topology(seed: int, lambda_topo: float = 0.15, max_epochs: int = 30):
    set_global_seed(seed)
    model_name = 'sasrec_topology'
    run_dir = get_run_dir(model_name, seed)
    logger = create_run_logger(run_dir / 'training.log', f'{model_name}_seed_{seed}')
    logger.info(f'BẮT ĐẦU HUẤN LUYỆN SASREC + TOPOLOGY | SEED = {seed} | LAMBDA = {lambda_topo}')
    
    item_mapping = load_item_mapping(PROCESSED_DIGI_DIR)
    max_internal_id = int(item_mapping['internal_item_id'].max())
    internal_to_raw = np.zeros(max_internal_id + 1, dtype=np.int64)
    internal_to_raw[item_mapping['internal_item_id'].to_numpy(dtype=np.int64)] = item_mapping['item_id'].to_numpy(dtype=np.int64)
    
    item_freqs = np.zeros(max_internal_id + 1, dtype=np.int64)
    item_freqs[item_mapping['internal_item_id'].to_numpy(dtype=np.int64)] = item_mapping['train_frequency'].to_numpy(dtype=np.int64)
    item_freqs_tensor = torch.tensor(item_freqs, dtype=torch.long, device=DEVICE)
    
    train_df = pd.read_parquet(PROCESSED_DIGI_DIR / CONFIG['data_files']['train_examples'])
    val_df = pd.read_parquet(PROCESSED_DIGI_DIR / CONFIG['data_files']['validation_examples'])
    test_df = pd.read_parquet(PROCESSED_DIGI_DIR / CONFIG['data_files']['test_examples'])
    test_last_df = pd.read_parquet(PROCESSED_DIGI_DIR / CONFIG['data_files']['test_last_item_examples'])
    test_events_full = pd.read_parquet(PROCESSED_DIGI_DIR / CONFIG['data_files']['test_events_full'])
    
    max_seq_len = int(CONFIG.get('max_seq_length', 50))
    train_dataset = SessionPrefixDataset(train_df, max_seq_length=max_seq_len)
    val_dataset = SessionPrefixDataset(val_df, max_seq_length=max_seq_len)
    test_dataset = SessionPrefixDataset(test_df, max_seq_length=max_seq_len)
    test_last_dataset = SessionPrefixDataset(test_last_df, max_seq_length=max_seq_len)
    
    train_loader = DataLoader(train_dataset, batch_size=int(CONFIG['training'].get('train_batch_size', 512)), shuffle=True, num_workers=0)
    
    model = build_model_from_config(CONFIG, item_mapping, DEVICE)
    cold_internal_ids = item_mapping.loc[item_mapping['is_controlled_cold'], 'internal_item_id'].to_numpy(dtype=np.int64)
    initial_cold_embeddings = model.item_embedding.weight.detach().cpu().numpy()[cold_internal_ids].copy()
    
    topo_criterion = TopologicalPersistenceLoss(tau=0.45, lambda_anisotropy=0.05, tail_threshold=20).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(CONFIG['training'].get('learning_rate', 0.001)))
    use_amp = bool(CONFIG['training'].get('use_amp', True)) and (DEVICE.type == 'cuda')
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    grad_clip = float(CONFIG['training'].get('grad_clip_norm', 5.0))
    
    best_score = -float('inf')
    best_epoch = 0
    epochs_no_improve = 0
    history_rows = []
    checkpoint_path = run_dir / 'checkpoint.pth'
    
    for epoch in range(1, max_epochs + 1):
        t0 = time.perf_counter()
        model.train()
        running_loss, running_topo = 0.0, 0.0
        total_samples = 0
        
        progress = tqdm(train_loader, desc=f'[SASREC+TOPO seed={seed}] Epoch {epoch:02d}/{max_epochs:02d}', leave=False)
        for batch in progress:
            item_seq = batch['item_seq'].to(DEVICE, non_blocking=True)
            item_seq_len = batch['item_seq_len'].to(DEVICE, non_blocking=True)
            targets = batch['target'].to(DEVICE, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model.full_sort_logits(item_seq, item_seq_len, mask_cold_and_pad=True)
                loss_ce = F.cross_entropy(logits, targets)
                
                session_vectors = model.encode_session(item_seq, item_seq_len)
                target_embs = model.item_embedding(targets)
                loss_topo = topo_criterion(session_vectors, target_embs, item_freqs_tensor, targets)
                
                loss = loss_ce + lambda_topo * loss_topo
            
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            
            bsz = item_seq.size(0)
            running_loss += float(loss_ce.item()) * bsz
            running_topo += float(loss_topo.item()) * bsz
            total_samples += bsz
            
        train_ce = running_loss / max(1, total_samples)
        train_topo = running_topo / max(1, total_samples)
        
        val_topk_ids, _ = predict_topk_for_dataset(model, val_dataset, DEVICE, batch_size=1024, topk=20)
        val_metrics = compute_ranking_metrics_from_predictions(val_topk_ids, val_dataset.targets, k=20)
        cur_score = float(val_metrics.get('mrr@20', 0.0))
        
        history_rows.append({
            'epoch': epoch, 'train_loss': train_ce, 'train_topo': train_topo,
            'val_recall@20': val_metrics.get('recall@20', 0), 'val_mrr@20': cur_score,
            'epoch_time_sec': time.perf_counter() - t0
        })
        logger.info(f'Epoch {epoch:02d} | Train CE: {train_ce:.4f} | Topo: {train_topo:.4f} | Val Recall@20: {val_metrics.get("recall@20", 0):.4f} | MRR: {cur_score:.4f}')
        
        if cur_score - best_score > 0.0001:
            best_score = cur_score
            best_epoch = epoch
            epochs_no_improve = 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'best_score': best_score}, checkpoint_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= 5:
                logger.info(f'Early Stopping tại Epoch {epoch}')
                break
                
    logger.info('Đánh giá Test Set với Checkpoint tốt nhất...')
    ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    full_weight_np = model.item_embedding.weight.detach().cpu().numpy()
    trained_cold_embeddings = full_weight_np[cold_internal_ids]
    max_cold_drift = float(np.max(np.abs(trained_cold_embeddings - initial_cold_embeddings)))
    item_embeddings = full_weight_np[1 : max_internal_id + 1].astype(np.float32)
    
    test_topk_ids, test_topk_scores = predict_topk_for_dataset(model, test_dataset, DEVICE, batch_size=1024, topk=20)
    test_metrics_main = compute_ranking_metrics_from_predictions(test_topk_ids, test_dataset.targets, k=20)
    test_stratified_df = compute_stratified_ranking_metrics(test_topk_ids, test_dataset.targets, test_dataset.target_groups, k=20)
    
    test_last_topk_ids, _ = predict_topk_for_dataset(model, test_last_dataset, DEVICE, batch_size=1024, topk=20)
    test_metrics_last_item = compute_ranking_metrics_from_predictions(test_last_topk_ids, test_last_dataset.targets, k=20)
    
    top20_df = build_top20_dataframe(test_dataset, test_topk_ids, test_topk_scores, internal_to_raw)
    top20_df.to_parquet(run_dir / 'top20.parquet', index=False, compression='zstd')
    
    bias_summary, item_rec_catalog, group_exposure_df = evaluate_top20_popularity_bias(top20_df, item_mapping, k=20)
    cold_diagnostic = compute_cold_start_test_diagnostic(test_metrics_main, test_events_full, len(test_dataset), k=20)
    
    save_yaml(CONFIG, run_dir / 'config.yaml')
    pd.DataFrame(history_rows).to_csv(run_dir / 'training_history.csv', index=False)
    np.save(run_dir / 'item_embeddings.npy', item_embeddings)
    
    embedding_mapping_df = item_mapping.copy()
    embedding_mapping_df.insert(0, 'embedding_row_index', np.arange(len(embedding_mapping_df)))
    embedding_mapping_df.to_csv(run_dir / 'item_embedding_mapping.csv', index=False)
    
    item_level_df = build_item_level_analysis_table(model_name, seed, item_embeddings, item_rec_catalog)
    item_level_df.to_parquet(ITEM_LEVEL_DIR / f'{model_name}_seed{seed}.parquet', index=False, compression='zstd')
    
    anisotropy = compute_anisotropy_spectrum(item_embeddings)
    
    res = {
        'model': model_name,
        'architecture': 'SASRec+Topology',
        'seed': int(seed),
        'device': str(DEVICE),
        'best_epoch': int(best_epoch),
        'best_validation_score': float(best_score),
        'monitor_metric': 'mrr@20',
        'cold_embedding_max_abs_drift': max_cold_drift,
        'test_warm_all_prefix': test_metrics_main,
        'test_warm_last_item': test_metrics_last_item,
        'test_warm_by_popularity_group': test_stratified_df.to_dict(orient='records'),
        'popularity_bias': bias_summary,
        'exposure_by_group': group_exposure_df.to_dict(orient='records'),
        'cold_start_diagnostic': cold_diagnostic,
        'anisotropy': anisotropy
    }
    save_json(res, run_dir / 'test_metrics.json')
    return res
print('Đã nạp xong Engine huấn luyện chuẩn mực 100%!')


In [ ]:
# ========================================================================
# 4. THỰC THI HUẤN LUYỆN TRÊN CẢ 2 RANDOM SEEDS (2024 & 2025)
# ========================================================================
results = {}
for s in [2024, 2025]:
    print(f'\n========== CHẠY SEED {s} ==========')
    results[s] = run_training_sasrec_topology(seed=s, lambda_topo=0.15, max_epochs=30)


In [ ]:
# ========================================================================
# 5. BẢNG TỔNG HỢP VÀ ĐỐI SÁNH KẾT QUẢ ĐẲNG CẤP
# ========================================================================
import json
rows = []
for s in [2024, 2025]:
    orig_f = RUNS_DIR / 'sasrec' / f'seed_{s}' / 'test_metrics.json'
    if orig_f.exists():
        with open(orig_f, 'r', encoding='utf-8') as f:
            d = json.load(f)
        tail_rec = 0.0
        for item in d.get('test_warm_by_popularity_group', []):
            if item.get('target_group') == 'rare':
                tail_rec = item.get('recall@20', 0)
        rows.append({
            'Model': 'SASRec Gốc (Baseline)',
            'Seed': s,
            'Overall Recall@20': d['test_warm_all_prefix'].get('recall@20', 0),
            'Overall NDCG@20': d['test_warm_all_prefix'].get('ndcg@20', 0),
            'Tail Recall@20 (Item B)': tail_rec,
            'Coverage@20 (%)': d['popularity_bias'].get('coverage_warm@20', 0) * 100,
            'Gini Index': d['popularity_bias'].get('gini_warm@20', 0)
        })
    topo_f = RUNS_DIR / 'sasrec_topology' / f'seed_{s}' / 'test_metrics.json'
    if topo_f.exists():
        with open(topo_f, 'r', encoding='utf-8') as f:
            d = json.load(f)
        tail_rec = 0.0
        for item in d.get('test_warm_by_popularity_group', []):
            if item.get('target_group') == 'rare':
                tail_rec = item.get('recall@20', 0)
        rows.append({
            'Model': 'SASRec + Topology (Đề xuất)',
            'Seed': s,
            'Overall Recall@20': d['test_warm_all_prefix'].get('recall@20', 0),
            'Overall NDCG@20': d['test_warm_all_prefix'].get('ndcg@20', 0),
            'Tail Recall@20 (Item B)': tail_rec,
            'Coverage@20 (%)': d['popularity_bias'].get('coverage_warm@20', 0) * 100,
            'Gini Index': d['popularity_bias'].get('gini_warm@20', 0)
        })

df_comp = pd.DataFrame(rows)
print('================ BẢNG ĐỐI SÁNH CHỈ SỐ ================')
display(df_comp)

if len(df_comp) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.barplot(data=df_comp, x='Seed', y='Tail Recall@20 (Item B)', hue='Model', ax=axes[0], palette=['#7f7f7f', '#2ca02c'])
    axes[0].set_title('Khả năng cứu Sản phẩm Hiếm (Tail-Recall@20)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Tail-Recall@20')

    sns.barplot(data=df_comp, x='Seed', y='Coverage@20 (%)', hue='Model', ax=axes[1], palette=['#7f7f7f', '#1f77b4'])
    axes[1].set_title('Độ bao phủ Danh mục (Catalog Coverage @20)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Coverage (%)')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'topology_tail_improvement_comparison.png', dpi=300)
    plt.show()
